# Thesis Documentation — Flow-Aware Temporal Pattern Mining for Multi-Stage NIDS

This notebook is `pipeline.ipynb` with a structured explanation inserted directly
above every logical code cell. The code cells are **unmodified**; only markdown
has been added, so this notebook executes exactly as the original does.

Stage numbers follow `Methodology_NIDS_.docx`. Each explanation records the
purpose of the step, the operations in execution order, the methods and equations
used, the important variables and config keys, the inputs and outputs, the link
to the corresponding methodology stage, and — under **Important Considerations** —
every known deviation from the methodology (labelled `Deviation:`) and every bug
found and fixed (labelled `Bug:`).

Deviations are cross-referenced to the identifiers used in `README.md`
(D1 … D23), so the thesis text and the repository documentation stay consistent.


# Flow-Aware Temporal Pattern Mining for Multi-Stage NIDS — CIC-IDS2017

End-to-end walkthrough of Stages 1–9 of the methodology, with every figure and
table displayed inline and written to `results/`.

Runs in **smoke mode** by default so the whole notebook executes in minutes. Set
`SMOKE = False` for a full run.

> Everything learned from data — imputer, scaler, encoder, correlation filter,
> the eight rankers, N\*, the calibrators, the fusion weights and τ_R — is fitted
> on the training partition, or on validation where the methodology says so.
> **The test partition is never used for fitting, selection or tuning.**

## 0. Setup

## 0. Environment, Configuration and Reproducibility Setup — Stage 0 (pre-stage)

### Purpose
Establish a deterministic, reproducible execution context before any data is
read. Every later stage depends on the configuration object and the random seed
fixed here, so this cell defines the experimental conditions of the whole run.

### What Happens in This Cell
1. Warnings are silenced and `TF_CPP_MIN_LOG_LEVEL=2` is set before TensorFlow is
   imported.
2. The repository root is prepended to `sys.path` so the nine project modules
   import.
3. `matplotlib` is switched to the inline backend, then the project modules are
   imported in dependency order (`utils`, `data_preprocessing`,
   `feature_selection`, `imbalance_handling`, `models`, `fusion`,
   `temporal_pattern_mining`, `evaluation`, `pipeline`).
4. `DATA_DIR` is resolved from `NIDS_DATA_DIR`, falling back to `data.data_dir`
   in `config.yaml`; `SMOKE` is read from `NIDS_SMOKE`.
5. A missing `DATA_DIR` raises `FileNotFoundError` rather than fabricating data.
6. `utils.load_config` builds `cfg`; `utils.set_global_seed(SEED)` seeds Python,
   NumPy, TensorFlow and PyTorch; `utils.configure_frameworks(cfg)` configures
   device and threading.
7. `CLASSES` and `CLASS_NAMES` fix the canonical class order.

### Method / Algorithm
Deterministic seeding across four RNG sources, and deep-merge configuration
resolution: `load_config` merges the `smoke:` block over the top level only when
`run.smoke_test` is true, so one file describes both regimes.

### Code Logic
`utils.load_config(path, overrides, apply_smoke)` deep-merges in a fixed order:
file → `overrides` → `smoke:` block. `set_global_seed` sets `PYTHONHASHSEED`,
`random.seed`, `np.random.seed`, `torch.manual_seed`,
`tf.keras.utils.set_random_seed` and `torch.backends.cudnn.deterministic`.
`configure_frameworks` sets `LOKY_MAX_CPU_COUNT`, enables TensorFlow GPU memory
growth and caps `torch.set_num_threads`, so TF and torch can share one process.
`CLASSES` is the integer index range over `cfg["labels"]["class_order"]`; this
ordering is what makes every model's probability matrix column-comparable in
Stage 7.

### Input and Output
**Inputs:** `config.yaml`; environment variables `NIDS_DATA_DIR`, `NIDS_SMOKE`.
**Config keys:** `run.smoke_test`, `run.output_dir`, `data.data_dir`,
`seed.primary`, `labels.class_order`.
**Outputs (in memory):** `cfg`, `SEED`, `CLASSES`, `CLASS_NAMES`, `OUT`, `env`.
**Outputs (disk):** `results/notebook/` is created.

### Research Methodology Connection
Implements the reproducibility requirement of Stage 8.3, which requires repeated
runs over several seeds with mean and standard deviation reported. Fixing the
seed here is the precondition for that.

### Important Considerations
- **Reproducibility:** `seed.multi_seed` lists seeds for multi-seed runs;
  `seed.enable_multi_seed` is false by default, so a single-seed run is reported
  without dispersion unless enabled.
- **Deviation (D14):** in smoke mode the `smoke:` block *raises* neural epoch
  budgets and learning rates rather than lowering them. A smoke epoch contains
  roughly two orders of magnitude fewer gradient steps, so a smaller budget would
  leave the LSTM and CNN underfitted and give a misleading quick result.
- **Runtime cost:** negligible, except that importing TensorFlow and PyTorch
  together costs several seconds and a few hundred MB of RSS.


In [ ]:
import os, sys, warnings, json
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import matplotlib.pyplot as plt

import utils
import data_preprocessing as dp
import feature_selection as fsel
import imbalance_handling as ih
import models as M
import fusion as fus
import temporal_pattern_mining as tpm
import evaluation as ev
import pipeline

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

# ---- configure the run -------------------------------------------------
# Point this at the folder holding the real CIC-IDS2017 daily CSVs, or export
# NIDS_DATA_DIR. Defaults to config.yaml's data.data_dir -- NOT a test fixture.
_cfg0 = utils.load_config("config.yaml", apply_smoke=False)
DATA_DIR = os.environ.get("NIDS_DATA_DIR", _cfg0["data"]["data_dir"])

# SMOKE = True subsamples heavily: use it to check the plumbing, never for
# results. Smoke artifacts carry their own cache signature and never mix with a
# full run. Default False = the full real dataset.
SMOKE = os.environ.get("NIDS_SMOKE", "0") == "1"
OUT = Path("results/notebook")
OUT.mkdir(parents=True, exist_ok=True)

if not Path(DATA_DIR).exists():
    raise FileNotFoundError(
        f"{DATA_DIR!r} does not exist. Set data.data_dir in config.yaml or export "
        f"NIDS_DATA_DIR to the folder holding the real CIC-IDS2017 daily CSVs.\n"
        f"To run on a generated fixture instead (plumbing checks only):\n"
        f"    python tests/synthetic_data.py --out_dir data/synthetic --rows 40000\n"
        f"    NIDS_DATA_DIR=data/synthetic NIDS_SMOKE=1 jupyter lab pipeline.ipynb")

cfg = utils.load_config("config.yaml", {"run": {"smoke_test": SMOKE,
                                                "output_dir": str(OUT)},
                                        "data": {"data_dir": DATA_DIR}})
SEED = utils.seeded(cfg)
utils.set_global_seed(SEED)
env = utils.configure_frameworks(cfg)

CLASSES = list(range(len(cfg["labels"]["class_order"])))
CLASS_NAMES = {i: n for i, n in enumerate(cfg["labels"]["class_order"])}

print(f"data_dir  : {DATA_DIR}")
print(f"smoke     : {cfg['run']['smoke_test']}")
print(f"seed      : {SEED}")
print(f"imbalance : {cfg['imbalance']['mode']}   split: {cfg['split']['strategy']}")
print(f"fusion    : {cfg['fusion']['mode']}")
print(f"env       : torch={env.get('torch')} tf={env.get('tensorflow')} gpu={env.get('gpu')}")

## 1. Config + data load — Stage 1

The daily CSVs are concatenated, column names stripped, the 12-hour afternoon
timestamps corrected from the file name, labels grouped per the Stage 1.3 table,
and the symmetric session key built (eq. 22, see README D5/D6).

## 1. Dataset Consolidation, Label Grouping and Session Reconstruction — Stage 1 and 2

### Purpose
Convert the raw CIC-IDS2017 daily CSV files into a single, schema-aligned,
time-ordered table with normalised class labels and reconstructed sessions. This
is the only cell that reads the raw dataset.

### What Happens in This Cell
`dp.prepare_dataset` executes Stages 1–2 in order:
1. `discover_csv_files` globs every CSV under `DATA_DIR` and sorts them by
   weekday and time of day.
2. `load_raw_dataset` reads each file, normalises column names, corrects the
   12-hour timestamps, repairs text-damaged numeric columns, and downcasts to
   `float32` before and after concatenation.
3. `group_labels` maps raw labels to the nine grouped classes and drops
   unlabeled rows.
4. `build_session_ids` constructs the session key.
5. `ColumnCleaner.run` removes duplicates, converts ±inf to NaN and applies the
   negative-value policy.
6. The cleaned frame is cached as parquet, and the class-distribution table and
   preprocessing report are written.

### Method / Algorithm
- **Timestamp correction:** CIC-IDS2017 stores times in 12-hour format without
  AM/PM. Files whose name contains an afternoon token have any parsed hour below
  `afternoon_shift_hour_max` advanced by `afternoon_hour_shift` (+12 h). Without
  this, afternoon traffic sorts before morning traffic and the Stage 3 time
  ordering is wrong.
- **Label grouping:** the Stage 1.3 table, applied after normalising punctuation
  and case, so `Web Attack \x96 Brute Force`, `Web Attack - XSS` and
  `Web Attack: SQL Injection` all reach `Web Attack`.
- **Session key:** equation (22), the symmetric key
  `{IP_min, IP_max, Port_min, Port_max, Protocol}`, so both directions of one
  conversation map to one session.

### Code Logic
`_normalise_label` lowercases and collapses non-alphanumeric runs before the
`labels.grouping` lookup; an unmapped *real* label raises rather than being
silently discarded. `_apply_session_timeout` splits each communication key into
sub-sessions whenever the inter-flow gap exceeds
`preprocessing.session_timeout_s` (1800 s) or the run reaches
`session_max_flows`. `COL_TIME`, `COL_ORDER`, `COL_SESSION`, `COL_LABEL`,
`COL_LABEL_RAW` and `COL_Y` are internal metadata columns and are never features.

### Input and Output
**Inputs:** the daily CSVs under `DATA_DIR`.
**Config keys:** `data.csv_glob`, `data.timestamp_column`,
`data.afternoon_file_tokens`, `data.negative_sentinel_features`,
`data.drop_unlabeled_rows`, `data.expect_min_rows`, `labels.grouping`,
`labels.class_order`, `preprocessing.session_key_mode`,
`preprocessing.session_timeout_s`.
**Outputs (memory):** `df`, `feature_columns`, `prep_report`.
**Outputs (disk):** `data/interim/clean__<signature>.parquet` and its
`.meta.json`; `results/notebook/class_distribution.csv`;
`results/notebook/preprocessing_report.json`; `class_distribution.png`.

### Research Methodology Connection
Implements Stage 1 (dataset consolidation, §1.2–1.3) and the split-independent
part of Stage 2 (§2.2 steps 1–3). Operations that learn parameters from data are
deliberately deferred to the post-split transformer.

### Important Considerations
- **Deviation (D1):** §1.2 states 8 files and §2.2 states 7. The code globs
  whatever is present and logs the count.
- **Deviation (D5):** `session_key_mode` defaults to `service_port`
  — `{IP_min, IP_max, Port_min, Protocol}` — because the client port in
  equation (22) is ephemeral and unique per flow, which would yield ≈1 flow per
  session and make the LSTM windows, the GNN graph and Stage 9 degenerate. Set
  `five_tuple` to reproduce equation (22) literally.
- **Deviation (D6):** the 1800 s idle timeout is not in the methodology. Without
  it a host pair active all week forms one session spanning the capture, which
  makes the Stage 3 time-blocked split impossible.
- **Deviation (D4):** the `MachineLearningCVE` release has no Flow ID, IP or
  Timestamp columns; surrogates are used and logged. The `TrafficLabelling`
  release is preferred.
- **Bug (fixed):** rows whose label is blank, NaN or an embedded header row from
  a concatenated CSV previously raised `unmapped raw labels: ['nan']`. They carry
  no supervision and are now dropped, counted in
  `prep_report["unlabeled_rows_dropped"]`.
- **Bug (fixed):** a single malformed cell made pandas infer a whole column as
  `object`, which silently disabled the `float32` downcast *and* the inf,
  negative-value and sentinel handling in `ColumnCleaner`. Columns that mostly
  parse as numbers are now coerced back with `errors="coerce"`.
- **Caching:** the parquet name embeds `utils.data_signature(cfg)` (mode, data
  folder, Dst-Port flag), so a smoke cache can never be reused by a full run.
- **Runtime cost:** the dominant cost of Stages 1–2; minutes on 2.83 M rows, then
  seconds on subsequent cached runs.


In [ ]:
df, feature_columns, prep_report = dp.prepare_dataset(
    cfg, data_dir=DATA_DIR, seed=SEED, out_dir=str(OUT), use_cache=True)

print(f"rows            : {len(df):,}")
print(f"candidate feats : {len(feature_columns)}")
print(f"sessions        : {df[dp.COL_SESSION].nunique():,}")
print(f"time range      : {df[dp.COL_TIME].min()}  ->  {df[dp.COL_TIME].max()}")
print(f"flows/session   : {len(df) / df[dp.COL_SESSION].nunique():.2f}")

dist = dp.class_distribution_table(df, cfg)
utils.save_table(dist, OUT / "class_distribution.csv")
# prepare_dataset returns early on a parquet cache hit and skips this,
# so write the Stage 2 report here as well.
utils.save_json(prep_report, OUT / "preprocessing_report.json")
dist

In [ ]:
ev.plot_class_distribution(dist, cfg, OUT / "class_distribution.png")
from IPython.display import Image, display
display(Image(str(OUT / "class_distribution.png")))

## 2. Preprocessing report and the split — Stages 2–3

`ColumnCleaner` does only what learns nothing from the data (dedup, inf→NaN,
negative-sentinel policy) so it may run before the split.
`TrainFittedPreprocessor` (median imputer + Min-Max + one-hot) is fitted on the
**training rows only** — it refuses anything else.

## 2. Preprocessing Report — Stage 2

### Purpose
Expose the Stage 2 cleaning decisions as auditable quantities rather than
side effects, so the data-quality claims in the thesis can be cited.

### What Happens in This Cell
The `prep_report` dictionary produced by Cell 1 is printed key by key, with long
lists truncated.

### Method / Algorithm
None — this cell only reports values computed in Cell 1.

### Code Logic
Reported keys include `rows_in` / `rows_out`, `duplicates_removed`,
`inf_values_to_nan`, `negative_values_to_nan`, `sentinel_features_preserved`,
`unlabeled_rows_dropped`, `n_feature_columns`, `identifier_columns_excluded`,
`use_dst_port` and `n_sessions`.

### Input and Output
**Input:** `prep_report`. **Output:** printed text; the same dictionary is saved
as `results/notebook/preprocessing_report.json` by Cell 1.

### Research Methodology Connection
Evidences Stage 2.2 steps 2–4 and the Stage 8.4 leakage control that identifier
and time columns are excluded from the feature matrix.

### Important Considerations
- **Deviation (D22):** `identifier_columns_excluded` includes `Destination Port`
  whenever `data.use_dst_port` is false, the default, because §8.4 flags it as a
  shortcut feature. Setting it true and re-running reports the dataset both ways.
- **Data quality:** a `duplicates_removed` count close to the row count of one
  input file indicates a duplicated source CSV and should be investigated before
  any result is trusted.


In [ ]:
print("--- preprocessing report (Stage 2) ---")
for k, v in prep_report.items():
    if k in ("classes",):
        continue
    if isinstance(v, list) and len(v) > 8:
        v = v[:8] + ["..."]
    print(f"  {k:32s} {v}")

## 3. Partitioning and Executable Leakage Controls — Stage 3 and 8.4

### Purpose
Fix which records may be used to learn each quantity, and verify that the
resulting partition actually satisfies the methodology's leakage constraints.

### What Happens in This Cell
`dp.make_split` dispatches on `cfg["split"]["strategy"]` (default
`time_blocked`), then `dp.split_distribution_table` builds the per-split class
counts and writes them to `split_distribution.csv`. The next cell runs
`utils.assert_no_leakage` and restates the per-class time ordering explicitly.

### Method / Algorithm
`time_blocked_split` works per class:
1. sessions of that class are ordered by start time;
2. two boundary *times* are taken as flow-count quantiles — `T1` at
   `train_frac`, `T2` at `train_frac + val_frac`;
3. a session joins train only if it **ends** by `T1`, a later block only if it
   **starts** after the previous boundary plus the gap; a session crossing a
   boundary is discarded into the gap.
Because a flow's timestamp is never earlier than its own session's start, this
yields the strict ordering train < validation < test required by §3.2.

### Code Logic
`_assign_units_in_time` performs the quantile cut. `SplitIndex` carries
`train`/`val`/`test` positional indices, `boundaries`, `dropped_gap` and a
`meta["assign_group"]` array recording which class's blocks placed each row.
`utils.assert_no_leakage` composes `assert_disjoint_splits`,
`assert_sessions_not_split` and `assert_no_time_leakage`; each raises
`LeakageError` rather than warning.

### Input and Output
**Inputs:** `df`, `cfg`, `SEED`.
**Config keys:** `split.strategy`, `split.train_frac`, `split.val_frac`,
`split.gap_frac`, `split.tiny_class_policy`,
`split.require_all_classes_in_all_splits`.
**Outputs (memory):** `split`, `split_table`, `report` (guard results).
**Outputs (disk):** `results/notebook/split_distribution.csv`.

### Research Methodology Connection
Implements Stage 3.2 (time-blocked per class, 70/15/15, whole sessions, a gap
between blocks) and makes the Stage 8.4 leakage controls executable rather than
declarative.

### Important Considerations
- **Deviation (D7):** §3.2 requires *both* that every class appear in all three
  partitions *and* that sessions stay whole. For a class with fewer separable
  sessions than partitions — Heartbleed (11 records) and Infiltration (36) — these
  are mutually unsatisfiable. The documented policy falls back to flow-level
  blocking for that class alone, records its sessions in
  `split.meta["exempt_sessions"]`, and the session-integrity guard honours that
  list rather than silently passing.
- **Deviation (D8):** sessions straddling a boundary are discarded; the count and
  percentage are logged and saved in `split_report.json`.
- **Leakage:** the `stratified_flow` strategy is deliberately leaky and exists
  only to quantify the optimism of the common protocol (see Cell 22).


In [ ]:
split = dp.make_split(df, cfg, SEED)

split_table = dp.split_distribution_table(df, split, cfg)
utils.save_table(split_table, OUT / "split_distribution.csv")

print(f"train/val/test  : {split.sizes()}")
print(f"gap discarded   : {len(split.dropped_gap)} flows "
      f"({100*len(split.dropped_gap)/len(df):.2f}%)")
print(f"exempt sessions : {len(split.meta.get('exempt_sessions', []))} "
      f"(tiny classes blocked at flow level - README D7)")
split_table

In [ ]:
# Leakage guards run as assertions, not as comments. Any violation raises.
report = utils.assert_no_leakage(
    split,
    session_ids=df[dp.COL_SESSION].to_numpy(),
    timestamps=df[dp.COL_TIME].to_numpy(),
    labels=df[dp.COL_LABEL].to_numpy(),
)
print("leakage checks passed:")
for k, v in report.items():
    print(f"  [OK] {k}")

# The time-blocked ordering, restated explicitly per class.
ts = df[dp.COL_TIME].to_numpy()
print("\nper-class block boundaries (train_max <= val_min <= test_min):")
groups = split.meta["assign_group"]
for g in sorted({x for x in np.unique(groups) if x}):
    rows = set(np.where(groups == g)[0].tolist())
    tr = [i for i in split.train if i in rows]
    va = [i for i in split.val if i in rows]
    te = [i for i in split.test if i in rows]
    if tr and te:
        print(f"  {str(g):14s} train_max={pd.Timestamp(ts[tr].max())}  "
              f"test_min={pd.Timestamp(ts[te].min())}  "
              f"ok={ts[tr].max() <= ts[te].min()}")

## 4. Train-Fitted Imputation, Scaling and Encoding — Stage 2.3 – 2.5

### Purpose
Learn every data-dependent transformation from the training partition alone and
apply it unchanged to validation and test, so no test information can reach a
fitted parameter.

### What Happens in This Cell
`dp.TrainFittedPreprocessor(cfg).fit(df, feature_columns, rows=split.train,
split=split)` fits the median imputer, the Min-Max scaler and the one-hot
encoder; `transform` is then applied separately to the three partitions,
producing `X_tr`, `X_va`, `X_te`.

### Method / Algorithm
- **Imputation:** per-feature median computed on training rows only (§2.3).
- **Normalisation:** Min-Max, equation (2), using the training minimum and
  maximum only.
- **Transformation:** one-hot encoding of `Protocol` fitted on training
  categories; a category unseen in training maps to all zeros (§2.5).

### Code Logic
`fit` stores `medians_`, `mins_`, `ranges_`, `constant_mask_` and `categories_`,
and records `fitted_rows_`. It calls `assert_fitted_on_train_only`, so passing a
non-training row raises `LeakageError`. Constant features (`max == min`) are
mapped to `preprocessing.constant_feature_value` instead of dividing by zero.
Values outside the training range are clipped to [0, 1] when
`preprocessing.minmax_out_of_range` is `"clip"`; the test minimum and maximum are
never consulted. `transform` never updates fitted statistics.

### Input and Output
**Inputs:** `df`, `feature_columns`, `split`.
**Config keys:** `preprocessing.imputer_strategy`, `preprocessing.scaler`,
`preprocessing.minmax_out_of_range`, `preprocessing.constant_feature_value`,
`data.protocol_encoding`.
**Outputs:** `pre`, `X_tr`, `X_va`, `X_te`, `y_tr`, `y_va`, `y_te`.

### Research Methodology Connection
Implements Stage 2.3–2.5 and the first Stage 8.4 control: the split precedes
imputation, scaling and encoding.

### Important Considerations
- **Deviation (D2):** the §2.2 table names "Median / Maximum Imputation" while
  §2.3 specifies and formulates the median. Median is implemented.
- **Leakage:** this is the single most common leakage point in published
  CIC-IDS2017 work — scaling before splitting leaks the test range into training.
  The guard makes that failure impossible rather than unlikely.


In [ ]:
# Fit the transformer on TRAIN only, then transform all three partitions.
pre = dp.TrainFittedPreprocessor(cfg).fit(df, feature_columns,
                                          rows=split.train, split=split)
X_tr = pre.transform(df, split.train)
X_va = pre.transform(df, split.val)
X_te = pre.transform(df, split.test)

y = df[dp.COL_Y].to_numpy()
y_tr, y_va, y_te = y[split.train], y[split.val], y[split.test]

print(f"X_train {X_tr.shape}   X_val {X_va.shape}   X_test {X_te.shape}")
print(f"constant features mapped to a constant: {int(pre.constant_mask_.sum())}")
print(f"one-hot columns added: {len(pre.output_columns_) - len(pre.numeric_columns_)}")
X_tr.describe().T[["min", "max", "mean", "std"]].head(8)

## 3. Feature selection — Stage 4

Redundancy filter → eight rankers (FeS Set-1 … Set-8) → frequency vote ψ ≥ η →
normalized-rank aggregation → N\* chosen on **validation** macro-F1.

## 5. Feature Selection by Fusion of Eight Independent Rankers — Stage 4

### Purpose
Reduce the CICFlowMeter feature space to a compact subset supported by several
independent criteria, so that no single ranker's inductive bias determines the
outcome.

### What Happens in This Cell
`fsel.run_feature_selection` executes Stage 4 in order:
1. `RedundancyFilter` removes near-zero-variance features
   (ε = `variance_epsilon`), duplicate columns, and one of each Spearman
   |ρ| > `spearman_threshold` pair, yielding `F_corr`;
2. the eight rankers each score every surviving feature;
3. a frequency vote and a normalized-rank aggregation produce the pool and its
   ordering;
4. `select_n_star` chooses `N*` on the validation partition;
5. artifacts are written and ranker scores cached.

### Method / Algorithm
The eight rankers (FeS Set-1 … Set-8) and their equations:
MI, k-NN estimator (3); Random Forest Gini importance (4); RBF-SVM permutation
importance measured on a held-out slice of TRAIN (5); mean |SHAP| of a
class-balanced LightGBM over rows **and** classes (6); |mean − median| of the
scaled feature (7); standard deviation of the scaled feature (8); XGBoost gain
importance (9); gradient×input importance of a class-weighted LSTM over session
windows (10).
Fusion: ψ(f) counts the top-K sets containing f and the pool is
`{f : ψ(f) ≥ η}` (12); `NR_m(f) = 1 − (rank − 1)/(K − 1)` (13);
`Score(f)` is the mean normalized rank over the eight rankers (14).

### Code Logic
`RANKERS` maps each name to its scoring function. `topk_set` discards scores ≤ 0
before taking the top K, per equation (11). `_sample` applies per-class capped
stratified sampling via `utils.stratified_subsample(..., min_per_class=...)`, so
rare classes survive every subsample. `fuse_rankings` returns one row per feature
with `psi_votes`, `score`, per-ranker membership and `in_pool`.
`joblib_cache` stores each ranker's scores under a key that embeds the data
signature.

### Input and Output
**Inputs:** `X_tr`, `y_tr`, `X_va`, `y_va`, plus training session ids and
timestamps for the LSTM ranker.
**Config keys:** `feature_selection.variance_epsilon`,
`.spearman_threshold`, `.top_k`, `.vote_threshold`, `.n_grid`, `.delta`,
`.rankers`, and the per-ranker sub-blocks.
**Outputs (memory):** `fs_result`.
**Outputs (disk):** `feature_votes_notebook.csv`, `ranker_scores_notebook.csv`,
`redundancy_dropped_notebook.csv`, `n_star_curve_notebook.csv`,
`feature_selection_notebook.json`, and cached ranker scores under
`data/interim/cache/`.

### Research Methodology Connection
Implements Stage 4.2 steps 1–5 and the algorithm of §4.7 steps 1–5, extending the
base paper from six rankers and top-10 to eight rankers and top-30 (§4.3).

### Important Considerations
- **Class imbalance:** handled *inside* the rankers through class weights and
  per-class capped stratified sampling (§4.4), never by resampling; SMOTE-ENN is
  deferred to Stage 5 so synthetic rows cannot influence which features are
  chosen.
- **Leakage:** every ranker sees training rows only. PI-SVM measures its
  permutation score on a slice held out *from TRAIN*, never validation or test
  (§4.5).
- **Deviation (D20):** PI-SVM is an auxiliary ranker and is deliberately absent
  from the Stage 6 classifiers (§4.4).
- **Bug (fixed):** PI-SVM built its scorer with
  `make_scorer(f1_score, average="macro")`, which inherits that function's binary
  default `pos_label=1`. When class 1 is absent from the SVM's sample — exactly
  what leave-one-attack-out does when holding out PortScan — sklearn raised
  `pos_label=1 is not a valid label`. A plain callable scorer replaced it.
- **Runtime cost:** the dominant Stage 4 cost. PI-SVM, SHAP-LGBM and the LSTM
  ranker are capped by `max_rows`, `explain_rows` and `max_windows` respectively;
  these caps are what keep Stage 4 tractable on 2.83 M rows.


In [ ]:
fs_result = fsel.run_feature_selection(
    X_tr, y_tr, X_va, y_va, cfg, SEED,
    session_ids_train=df[dp.COL_SESSION].to_numpy()[split.train],
    times_train=df[dp.COL_TIME].to_numpy()[split.train],
    out_dir=str(OUT), cache_tag="notebook",
)

print(f"F_corr after redundancy removal : {len(fs_result.f_corr)}")
print(f"pool P (psi >= {cfg['feature_selection']['vote_threshold']})            "
      f": {len(fs_result.pool)}")
print(f"N* chosen on validation         : {fs_result.n_star}")
print(f"\nF* = {fs_result.f_star}")

## 6. The Eight Candidate Sets and the Vote Table — Stage 4.4 – 4.6

### Purpose
Expose the intermediate evidence behind `F*` — which ranker nominated which
feature, and how the agreement translated into an ordering — so the selection is
auditable rather than a black box.

### What Happens in This Cell
The first cell rebuilds each ranker's top-K set with `fsel.topk_set` and displays
them side by side. The second shows the head of `fs_result.votes_table` with the
vote count, the fused score, pool membership and the per-ranker `in_*` flags.

### Method / Algorithm
Display only; the quantities were computed in Cell 5 by equations (11)–(14).

### Code Logic
`fsel.FES_SET_LABELS` maps each ranker key to its methodology label
("FeS Set-1 (MI)" … "FeS Set-8 (LSTM)"). `votes_table` columns: `feature`,
`psi_votes` (ψ), `score` (fused Score), `in_pool` (ψ ≥ η), and `in_<ranker>` /
`nr_<ranker>` per ranker.

### Input and Output
**Inputs:** `fs_result.ranker_scores`, `fs_result.votes_table`,
`cfg["feature_selection"]["top_k"]`. **Output:** rendered tables; the same data is
in `feature_votes_notebook.csv`.

### Research Methodology Connection
Shows §4.4's eight perspectives and §4.6's agreement levels — support from 8
rankers is very strong agreement, 6 strong, 4 moderate, 1 isolated.

### Important Considerations
- Interpretive: the statistical rankers (DMM, SD) measure distributional shape,
  not class relevance, so §4.5 cautions that they act as supporting evidence and
  never decide the subset alone. Their appearance high in a single column is
  therefore not by itself evidence of usefulness.


In [ ]:
# The eight top-K sets (FeS Set-1 ... Set-8)
K = cfg["feature_selection"]["top_k"]
sets = {fsel.FES_SET_LABELS[m]: fsel.topk_set(s, fs_result.f_corr, K)
        for m, s in fs_result.ranker_scores.items()}
pd.DataFrame({k: pd.Series(v[:12]) for k, v in sets.items()}).fillna("")

In [ ]:
fs_result.votes_table.head(20)[    ["feature", "psi_votes", "score", "in_pool"]    + [c for c in fs_result.votes_table.columns if c.startswith("in_")]]

## 7. Selection Figures, Subset Size N* and Stability — Stage 4.6 and 8.3

### Purpose
Visualise cross-ranker agreement, justify the chosen subset size on held-out
data, and quantify how stable the selection is under resampling.

### What Happens in This Cell
Three figures are rendered (vote heatmap, rank-score bars, `N*` curve), then the
validation search table is displayed, and the stability table and plot are shown
when `feature_selection.stability.enabled` is true.

### Method / Algorithm
- **N\*:** equation (15). Candidate sizes `N ∈ n_grid` are scored with a
  class-balanced LightGBM on the **validation** partition, and `N*` is the
  *smallest* N whose macro-F1 is within `delta` of the best.
- **Stability:** equation (28), `Stability(f) = selected runs / total runs` over
  repeated stratified subsamples of TRAIN.

### Code Logic
`select_n_star` also records weighted F1, minority recall, PR-AUC and inference
time per candidate size, so the size choice can be defended on more than one
axis. `compute_stability` re-runs all eight rankers on `n_runs` subsamples of
fraction `subsample_frac`; it is computed **after** `F*` is fixed and never
alters it.

### Input and Output
**Inputs:** `fs_result.votes_table`, `.n_star_curve`, `.stability`, `.n_star`.
**Outputs (disk):** `feature_votes_heatmap.png`, `feature_rank_scores.png`,
`n_star_curve.png`, `feature_stability.png`, `feature_stability_notebook.csv`.

### Research Methodology Connection
Implements §4.6 equation (15) and the stability half of §8.3.

### Important Considerations
- **Leakage:** `N*` is chosen on validation, never on test — one of the Stage 8.4
  controls.
- **Runtime cost:** stability re-runs all eight rankers `n_runs` times and is the
  single most expensive diagnostic in Stage 4. It is disabled for the
  leakage-gap comparison in Cell 22 precisely because it cannot affect the metric
  compared there.


In [ ]:
for fn, path in [
    (lambda: ev.plot_feature_votes(fs_result.votes_table, cfg, OUT / "feature_votes_heatmap.png"),
     OUT / "feature_votes_heatmap.png"),
    (lambda: ev.plot_rank_scores(fs_result.votes_table, cfg, OUT / "feature_rank_scores.png"),
     OUT / "feature_rank_scores.png"),
    (lambda: ev.plot_n_star_curve(fs_result.n_star_curve, fs_result.n_star, cfg,
                                  OUT / "n_star_curve.png"), OUT / "n_star_curve.png"),
]:
    fn()
    display(Image(str(path)))

In [ ]:
print("N* search on the VALIDATION partition:")
display(fs_result.n_star_curve)

if fs_result.stability is not None:
    ev.plot_stability(fs_result.stability, cfg, OUT / "feature_stability.png")
    display(Image(str(OUT / "feature_stability.png")))
    display(fs_result.stability.head(12))

## 8. Freezing the Optimal Feature Set F* — Stage 4.7 step 8

### Purpose
Fix the selected subset once and apply it unchanged everywhere, so that training,
validation and test all see identical columns in identical order.

### What Happens in This Cell
`fs_result.apply` is called on each of `X_tr`, `X_va`, `X_te`, producing `F_tr`,
`F_va`, `F_te`; `utils.assert_feature_mask_frozen` then verifies the test columns.

### Method / Algorithm
Equation (16): `F* = Top_{N*}(P ; Score)`, `X* = X[:, F*]`.

### Code Logic
`apply` raises `KeyError` if a selected feature is missing and calls
`assert_feature_mask_frozen`, which compares the expected and actual column lists
*including order*. Order matters because the Stage 6 CNN consumes features as an
ordered vector (equation 19).

### Input and Output
**Inputs:** `X_tr`, `X_va`, `X_te`, `fs_result.f_star`.
**Outputs:** `F_tr`, `F_va`, `F_te`.

### Research Methodology Connection
Implements §4.7 step 8 — freeze `F*`, apply to train, validation and test as a
transform only — and the Stage 8.4 control that `F*` is frozen.

### Important Considerations
- Re-ordering the columns between stages would silently change the CNN's input
  topology while leaving every tree model unaffected, which is why the guard
  compares order and not just membership.


In [ ]:
# Freeze F* and apply it unchanged to all three partitions.
F_tr = fs_result.apply(X_tr, "train")
F_va = fs_result.apply(X_va, "validation")
F_te = fs_result.apply(X_te, "test")
utils.assert_feature_mask_frozen(fs_result.f_star, list(F_te.columns), "test")
print(f"F* applied unchanged: {F_tr.shape}, {F_va.shape}, {F_te.shape}")

## 4. Imbalance handling — Stage 5

Training partition only, **after** feature selection, so synthetic rows cannot
influence which features were chosen. Validation and test are never resampled.

## 9. Imbalance Handling: Class Weighting and SMOTE-ENN — Stage 5

### Purpose
Compensate for the extreme class imbalance of CIC-IDS2017 without letting the
correction touch the evaluation partitions.

### What Happens in This Cell
`ih.apply_imbalance_handling` dispatches on `imbalance.mode`, then
`ih.assert_eval_sets_untouched` asserts that validation and test row counts are
unchanged. The report table is written and the before/after counts are plotted.

### Method / Algorithm
- **Class weighting:** `w_c = N / (C · N_c)` (§5.2), supplied as sklearn
  `class_weight`, XGBoost sample weights, Keras `class_weight` or a weighted
  torch loss.
- **SMOTE-ENN:** SMOTE oversampling followed by Edited Nearest Neighbours
  cleaning, applied to the training partition only and **after** feature
  selection.

### Code Logic
The documented minimum-sample policy: `k_neighbors = min(cfg_k, n_c − 1)`, since
SMOTE interpolates between a sample and its k same-class neighbours. A class with
fewer than `min_samples_for_smote` training samples cannot support meaningful
interpolation and takes `small_class_fallback` (`random_oversample` by default);
Heartbleed and Infiltration reach this path. `protect_rare_classes` restores the
originals of any class ENN removes entirely. Every decision is recorded in
`ResampleReport`.

### Input and Output
**Inputs:** `F_tr`, `y_tr`, `CLASSES`.
**Config keys:** `imbalance.mode`, `imbalance.smote_enn.k_neighbors`,
`.min_samples_for_smote`, `.small_class_fallback`, `.majority_cap`,
`.target_strategy`, `.max_oversample_ratio`, `.protect_rare_classes`.
**Outputs:** `X_bal`, `y_bal`, `class_weights`, `resample_report`;
`imbalance_report.csv`, `imbalance_before_after.png`.

### Research Methodology Connection
Implements Stage 5.2 and its requirement that the two techniques be comparable as
branches (class weighting alone, SMOTE-ENN alone, both). The notebook runs the
`both` configuration, which is branch C.

### Important Considerations
- **Leakage:** validation and test are never resampled; the assertion makes this
  a hard failure rather than a convention.
- **Deviation (D9):** imblearn's default raises every class to the majority
  count, which on CIC-IDS2017 manufactures roughly 2.27 M synthetic Heartbleed
  rows from 11 real ones. Measured on the development fixture, that drove the CNN
  to 0.75 training accuracy against 0.14 validation accuracy. `target_strategy:
  capped` limits a class to `max_oversample_ratio` × its real count and leaves the
  residual imbalance to class weighting — which is precisely why branch C
  combines the two. Set `full` for the literal reading.
- **Interpretation:** `majority_cap` changes the benign/attack ratio the models
  see, so results are not directly comparable across different cap values.


In [ ]:
n_val_before, n_test_before = len(F_va), len(F_te)

X_bal, y_bal, class_weights, resample_report = ih.apply_imbalance_handling(
    F_tr, y_tr, cfg, SEED, classes=np.array(CLASSES))

# Assert the evaluation partitions came through untouched.
ih.assert_eval_sets_untouched(n_val_before, len(F_va), n_test_before, len(F_te))
print("[OK] validation and test were not resampled\n")

table = resample_report.to_frame(CLASS_NAMES)
utils.save_table(table, OUT / "imbalance_report.csv")
display(table)

print("\npolicy notes:")
for n in resample_report.notes:
    print("  -", n)
if class_weights:
    print("\nclass weights w_c = N / (C * N_c):")
    print({CLASS_NAMES[k]: round(v, 3) for k, v in class_weights.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
idx = np.arange(len(table))
ax.bar(idx - 0.2, table["before"], 0.4, label="before")
ax.bar(idx + 0.2, table["after"], 0.4, label="after")
ax.set_xticks(idx, table["class"], rotation=30, ha="right")
ax.set_yscale("log"); ax.set_ylabel("training rows (log)")
ax.set_title(f"Stage 5: class counts before/after ({cfg['imbalance']['mode']})")
ax.legend(); ax.grid(axis="y", alpha=0.3)
utils.save_fig(fig, OUT / "imbalance_before_after.png", cfg["run"]["figure_dpi"], close=False)
plt.show()

## 5. Models — Stage 6

RF, XGBoost, LSTM, 1-D CNN and the temporal GNN. The LSTM and GNN train on the
un-resampled ordered flows with class weighting (README D10).

## 10. Training the Five Heterogeneous Classifiers — Stage 6

### Purpose
Train five models with complementary inductive biases on the frozen feature set,
each emitting a class-probability vector for Stage 7.

### What Happens in This Cell
Per-partition context (`session_ids`, `times`, `hosts`) is built by
`pipeline._context`. The flow-level models (RF, XGBoost, CNN) train on the
resampled `X_bal`; the sequence and graph models (LSTM, GNN) train on the
un-resampled `F_tr` with class weighting. Training curves are then plotted.

### Method / Algorithm
- **RF (§6.3):** 200 trees, `max_features=sqrt(k)`, Gini, class weights.
- **XGBoost (§6.4):** `multi:softprob`, L1/L2 regularisation, early stopping on
  validation.
- **LSTM (§6.5):** input is a window of up to L = 10 consecutive flows of one
  session, shape (L, k), left-padded and masked; one window per flow, so the
  output aligns one-to-one with the flow-level models.
- **1-D CNN (§6.6):** exactly the §6.6 architecture table —
  `Conv1D(64,k3,same) → ReLU → BatchNorm → MaxPool(2) → Conv1D(128,k3) → ReLU →
  MaxPool(2) → Flatten → Dense(128) → Dropout(0.3) → Softmax(C)`. Features arrive
  ordered by descending fusion Score (equation 19).
- **GNN (§6.7):** a 2-layer GCN over a time-causal communication graph,
  equation (23), `H' = σ(D̃^-1/2 Ã D̃^-1/2 H W)`.

### Code Logic
Every model subclasses `BaseModel` and implements `fit`, `predict_proba`, `save`
and `load`. `_align` maps each model's own class order onto the canonical order,
emitting a zero column for a class the model never saw and renormalising — which
is what makes the five probability matrices column-comparable in Stage 7.
`build_session_graph` builds session nodes whose attributes are the mean of `F*`
over the session's flows, with edges joining sessions that share a host,
directed earlier → later.

### Input and Output
**Inputs:** `X_bal`, `y_bal`, `F_va`, `y_va`, `class_weights`, contexts.
**Config keys:** `models.enabled`, `models.rf`, `models.xgb`, `models.lstm`,
`models.cnn`, `models.gnn`, `imbalance.apply_to_sequence_models`.
**Outputs:** `trained` (name → model), `training_curves.png`.

### Research Methodology Connection
Implements Stage 6.3–6.7 and §6.9: each model produces `P_m(Y | X*)`, equation
(17), which are **not** averaged but passed to Stage 7.

### Important Considerations
- **Deviation (D3):** the §6.6 prose diagram says "Global pooling" while the §6.6
  architecture table specifies `MaxPooling1D(2) → Flatten` with a worked shape
  example. The table is implemented, as the more specific statement.
- **Deviation (D10):** a SMOTE sample is an interpolated point with no session,
  timestamp or host, so it cannot occupy a position in a flow window or carry a
  graph edge. With `apply_to_sequence_models: false` the LSTM and GNN train on the
  original ordered flows with class weighting instead.
- **Deviation (D11):** §6.7 permits "hosts or endpoints" as nodes; **sessions**
  are used, because the label and `F*` aggregate naturally to a session whereas a
  host carries many unrelated labels. The node target is the session's
  **majority** class (the rarest class is used elsewhere, for split placement,
  where the goal is the opposite).
- **Deviation (D12):** measured on the development fixture, host-sharing edges
  joined same-class sessions 39.6 % of the time against a 38.5 % chance baseline
  — essentially no class signal — and the GCN's neighbourhood averaging destroyed
  the node's own evidence. `models.gnn.residual` adds a linear skip from the raw
  node features to the logits and `edge_max_time_gap_s` keeps only
  temporally-local edges. Equation (23) remains the propagation rule.
- **Deviation (D13):** Keras defaults BatchNorm momentum to 0.99, which needs
  thousands of updates for the inference-time moving averages to converge. On
  short schedules the CNN trained on batch statistics and inferred on
  near-initialisation ones. `models.cnn.batchnorm_momentum: 0.9` fixes this.
- **Bug (fixed):** the neural models originally sized their softmax over all
  canonical classes and so assigned probability to classes absent from training —
  which breaks leave-one-attack-out by construction. They now emit output units
  for seen classes only, as the tree models already did.
- **Runtime cost:** the dominant Stage 6 cost; `models.predict_chunk_size` bounds
  prediction memory on 2.83 M rows.


In [ ]:
ctx_tr = pipeline._context(df, split.train)
ctx_va = pipeline._context(df, split.val)
ctx_te = pipeline._context(df, split.test)

seq_models = set(cfg["imbalance"].get("sequence_models", ["lstm", "gnn"]))
resampled = len(X_bal) != len(F_tr)
split_training = resampled and not cfg["imbalance"].get("apply_to_sequence_models", False)

flow_cfg = utils.deep_merge(cfg, {"models": {"enabled": [
    m for m in cfg["models"]["enabled"] if not (split_training and m in seq_models)]}})
trained = M.train_all_models(X_bal, y_bal, F_va, y_va, flow_cfg, CLASSES, SEED,
                             class_weights=class_weights,
                             train_context=ctx_tr, val_context=ctx_va)

if split_training:
    seq_cfg = utils.deep_merge(cfg, {"models": {"enabled": [
        m for m in cfg["models"]["enabled"] if m in seq_models]}})
    w_seq = ih.compute_class_weights(y_tr, cfg, np.array(CLASSES))
    trained.update(M.train_all_models(F_tr, y_tr, F_va, y_va, seq_cfg, CLASSES, SEED,
                                      class_weights=w_seq,
                                      train_context=ctx_tr, val_context=ctx_va))

print("trained:", sorted(trained))

In [ ]:
# Training curves for the Keras models and the GNN.
curves = {n: m.history_ for n, m in trained.items() if hasattr(m, "history_")}
if curves:
    fig, axes = plt.subplots(1, len(curves), figsize=(4.6 * len(curves), 3.6), squeeze=False)
    for ax, (name, h) in zip(axes[0], curves.items()):
        ax.plot(h["loss"], label="train loss")
        if "val_loss" in h:
            ax.plot(h["val_loss"], label="val loss")
        ax.set_title(f"{name} training"); ax.set_xlabel("epoch")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout()
    utils.save_fig(fig, OUT / "training_curves.png", cfg["run"]["figure_dpi"], close=False)
    plt.show()

## 11. Probability Generation and Per-Model Metrics — Stage 6.9

### Purpose
Produce the calibrated-ready probability matrices for validation and test, verify
their mathematical validity, and record the efficiency figures Stage 8.1 requires.

### What Happens in This Cell
`M.predict_all` runs each model over `F_va` and `F_te`;
`utils.assert_probabilities_valid` checks every matrix; `M.model_stats_table`
collects efficiency figures; a per-model metric table is displayed.

### Method / Algorithm
Equation (17) per model. Validation checks that each row is non-negative, finite
and sums to one.

### Code Logic
`assert_probabilities_valid` raises `LeakageError` on a non-simplex row, so a
model that silently emits logits or an unnormalised vector cannot reach fusion.
`model_stats_table` reports `train_time_s`, `inference_ms_per_flow`,
`n_parameters`, `peak_memory_mb`, and for the GNN the backend actually used and
the node and edge counts.

### Input and Output
**Inputs:** `trained`, `F_va`, `F_te`, contexts.
**Outputs:** `val_proba`, `test_proba`, `stats`; `model_efficiency.csv`.

### Research Methodology Connection
Implements §6.9 and the efficiency group of §8.1 — feature-selection time,
training time, inference latency per flow, number of features, memory.

### Important Considerations
- **Deviation (D23):** the GNN uses `torch_geometric.nn.GCNConv` when importable
  and otherwise an equivalent dense normalised-adjacency implementation of
  equation (23); the backend used is recorded in `model_efficiency.csv`.
- **Evaluation validity:** inference latency is measured on batched prediction
  and is therefore a lower bound for a streaming deployment.


In [ ]:
val_proba = M.predict_all(trained, F_va, ctx_va)
test_proba = M.predict_all(trained, F_te, ctx_te)

for name, p in test_proba.items():
    utils.assert_probabilities_valid(p, name)
print("[OK] every model's probabilities form a simplex over the canonical class order\n")

stats = M.model_stats_table(trained)
utils.save_table(stats, OUT / "model_efficiency.csv")
display(stats)

per_model = ev.metrics_to_frame([
    ev.evaluate_predictions(y_te, p, CLASSES, cfg, CLASS_NAMES, label=n)
    for n, p in test_proba.items()])
per_model[["label", "accuracy", "macro_f1", "weighted_f1",
           "roc_auc_ovr_macro", "ece"]].sort_values("macro_f1", ascending=False)

## 6. Calibration — Stage 7.1

Isotonic/Platt for the tree and LSTM models, temperature scaling for the CNN and
GNN. **Fitted on the validation partition only.**

## 12. Probability Calibration on the Validation Partition — Stage 7.1

### Purpose
Make the five models' probabilities comparable on a common scale before they are
weighted against each other, so that a confidently wrong model cannot dominate
the fusion.

### What Happens in This Cell
`fus.fit_calibrators` fits one calibrator per model on validation;
`fus.apply_calibrators` applies them to validation and test; ECE and Brier before
and after are tabulated; reliability diagrams are drawn.

### Method / Algorithm
§7.1 prescribes isotonic or Platt-type calibration for the tree models and
temperature scaling for the CNN and GNN. `TemperatureScaler` fits a single
parameter T minimising validation NLL of `softmax(log p / T)`; `PerClassCalibrator`
applies one-vs-rest isotonic or Platt regression per class and renormalises the
row to the simplex.

### Code Logic
`fusion.calibration` maps each model name to its method. Calibrators are fitted
on `y_va` only and then frozen; `apply_calibrators` learns nothing. A class that
is absent or constant in validation is left uncalibrated rather than fitted on
degenerate data. `expected_calibration_error` uses
`evaluation.ece_bins` equal-width bins over the predicted-class confidence.

### Input and Output
**Inputs:** `val_proba`, `y_va_idx`, `test_proba`.
**Config keys:** `fusion.calibration.*`, `evaluation.ece_bins`.
**Outputs:** `calibrators`, `val_cal`, `test_cal`; `reliability_<model>.png`.

### Research Methodology Connection
Implements Stage 7.1, the precondition for §7.3's reliability weighting — weights
compare like with like only if the probabilities are calibrated.

### Important Considerations
- **Leakage:** calibrators are fitted on validation only, never on test — a
  Stage 8.4 control.
- **Evaluation validity:** calibration changes ECE and Brier substantially while
  often leaving argmax accuracy unchanged, because isotonic regression is
  monotone per class. Both are reported for that reason.


In [ ]:
y_va_idx = np.array([CLASSES.index(v) for v in y_va])
calibrators = fus.fit_calibrators(val_proba, y_va_idx, cfg)
val_cal = fus.apply_calibrators(val_proba, calibrators)
test_cal = fus.apply_calibrators(test_proba, calibrators)

rows = []
for name in val_proba:
    rows.append({
        "model": name,
        "calibrator": type(calibrators.get(name)).__name__ if name in calibrators else "none",
        "ECE before": ev.expected_calibration_error(val_proba[name], y_va_idx),
        "ECE after": ev.expected_calibration_error(val_cal[name], y_va_idx),
        "Brier before": ev.brier_score(val_proba[name], y_va_idx),
        "Brier after": ev.brier_score(val_cal[name], y_va_idx),
    })
pd.DataFrame(rows).round(4)

In [ ]:
for name in list(val_proba)[:3]:
    ev.plot_reliability_diagram(val_proba[name], val_cal.get(name), y_va_idx, cfg,
                                OUT / f"reliability_{name}.png",
                                title=f"Reliability diagram - {name} (validation)")
    display(Image(str(OUT / f"reliability_{name}.png")))

## 7. Adaptive evidence fusion — Stage 7

`q_i(t) = λ1·Q_i + λ2·C_i(t) + λ3·K_i(t)`, `w_i(t) = q_i(t) / Σ q_j(t)`.
First the Stage 9 evidence that the fusion consumes.

## 13. Temporal Evidence and Fitting the Reliability Weights — Stage 9 and 7.3

### Purpose
Mine the sequential and transition evidence that the fusion consumes, then fit
every fusion parameter — baseline reliabilities, the λ balance and the alert
threshold — on the validation partition.

### What Happens in This Cell
`tpm.run_temporal_mining` mines TRAIN sequences and builds the attack-state
graph. `temporal.evidence_for` produces per-flow `SP_t`, `TC_t` and availability
for validation and test. `fus.previous_states` supplies the preceding event per
session for the context term. `fus.EvidenceFusion(...).fit(...)` then fits
`Q_i`, the λ triple and `τ_R`.

### Method / Algorithm
- **Q_i (§7.3 step 2):** validation macro-F1 per model. For `SP` and `TC`, the
  validation ROC-AUC of each used **alone** as an attack score — a discriminative
  measure, because the plain average of `SP_t` would reward high values whether
  or not they are correct.
- **Context reliability (§7.3 step 3), equation (25):**
  `q_i(t) = λ₁·Q_i + λ₂·C_i(t) + λ₃·K_i(t)`, with
  `C_i(t) = 1 − H(P̂_i(t)) / log C` the normalised-entropy confidence and
  `K_i(t)` the attack-state-graph probability of the state model *i* predicts
  given the previous state.
- **Availability:** `q_S(t) = Q_S · a_S(t)`, where `a(t) = 0` when the session has
  fewer than `fusion.min_events_for_temporal` events, so unavailable evidence
  receives exactly zero weight.
- **τ_R (§7.3 step 5):** chosen on validation to maximise macro-F1 or to meet a
  target false-alarm rate.

### Code Logic
`fit` tunes λ over a simplex grid of step `tune_lambdas.grid_step` by validation
macro-F1. `_fit_threshold` sweeps `threshold.grid_points` thresholds. The
attack-state graph applies Laplace smoothing (`laplace_alpha`), so an unseen
transition lowers a model's reliability without zeroing its weight entirely.

### Input and Output
**Inputs:** `df`, `split.train`, `val_cal`, `y_va`.
**Config keys:** `temporal.*`, `fusion.lambdas`, `fusion.tune_lambdas`,
`fusion.threshold`, `fusion.min_events_for_temporal`, `fusion.mode`.
**Outputs:** `temporal`, `ev_va`, `ev_te`, `prev_va`, `prev_te`, `fusion`;
`fp_growth_itemsets.csv`, `prefixspan_patterns.csv`, `attack_state_graph.json`,
`attack_state_graph.graphml`.

### Research Methodology Connection
Implements the Stage 9 extension and Stage 7.3 steps 1–3 and 5.

### Important Considerations
- **Deviation (D15–D19):** §7.2–7.3 consume `SP_t`, `TC_t` and `K_i(t)` but
  Stage 9 is named without being specified. The implemented reading is: the event
  alphabet is a discretised flow state by default (`temporal.event_definition`),
  keeping the sequence model independent of the labels; `SP_t` is the maximum
  support among mined TRAIN patterns appearing as an ordered subsequence of the
  session, normalised by the largest TRAIN support; `TC_t` is the mean transition
  probability along the session's own path through `G_T`, scaled by the number of
  states and clipped to [0, 1]. All choices are configurable.
- **Deviation (D23):** FP-Growth uses `mlxtend` and PrefixSpan the `prefixspan`
  package when importable; equivalent implementations are bundled and used
  otherwise, and the log records which ran.
- **Leakage:** patterns and `G_T` are mined on TRAIN rows only; λ, `Q_i` and
  `τ_R` are fitted on validation only.


In [ ]:
temporal = tpm.run_temporal_mining(df, split.train, cfg, out_dir=str(OUT),
                                   class_names=CLASS_NAMES)
use_temporal = cfg["fusion"]["mode"] == "with_temporal"

ev_va = temporal.evidence_for(df, split.val, cfg) if use_temporal else None
ev_te = temporal.evidence_for(df, split.test, cfg) if use_temporal else None

events_va = temporal.discretiser.transform(df.iloc[split.val]) if temporal.discretiser else None
events_te = temporal.discretiser.transform(df.iloc[split.test]) if temporal.discretiser else None
prev_va = fus.previous_states(df, split.val, cfg, events_va)
prev_te = fus.previous_states(df, split.test, cfg, events_te)

fusion = fus.EvidenceFusion(cfg=cfg, classes=CLASSES, class_names=CLASS_NAMES,
                            calibrators=calibrators, use_temporal=use_temporal,
                            graph=temporal.graph)
fusion.fit(val_cal, y_va, prev_state=prev_va,
           sp=ev_va["sp"] if ev_va else None,
           tc=ev_va["tc"] if ev_va else None,
           availability=ev_va["availability"] if ev_va else None,
           graph=temporal.graph)

print(f"baseline reliabilities Q : "
      f"{ {k: round(v, 4) for k, v in fusion.Q.items()} }")
print(f"lambdas (tuned on val)   : {tuple(round(l, 2) for l in fusion.lambdas)}")
print(f"tau_R (tuned on val)     : {fusion.tau_R:.4f}")

## 14. Reliability-Aware Adaptive Fusion of the Evidence — Stage 7.2

### Purpose
Combine the five calibrated model outputs and the two temporal evidence sources
into one fused class distribution and one risk score, with weights that change
per session according to measurable reliability.

### What Happens in This Cell
`fusion.fuse` computes the per-source reliabilities, normalises them into
weights, and produces the fused probabilities and risk score. Both invariants are
asserted, artifacts are saved, and the mean weight per source is displayed and
plotted.

### Method / Algorithm
- Weights, equation (26): `w_i(t) = q_i(t) / Σ_j q_j(t)`.
- Risk, equation (24):
  `R_t = Σ_i w_i(t)·P̂_i,t + w_S(t)·SP_t + w_T(t)·TC_t`, with
  `Σ_i w_i(t) + w_S(t) + w_T(t) = 1`.
- Fused class probabilities, equation (27):
  `P_fused(c|x) = Σ_i w_i(x)·P̂_i(c|x)`, `ŷ = argmax_c P_fused`.

### Code Logic
A model's scalar attack evidence is taken as `1 − P̂_i(Benign)`, which is the
document's own definition for the CNN in §6.6. The fused *class* probabilities
are formed from the model terms alone and renormalised, because `SP` and `TC`
carry no class distribution. `assert_weights_sum_to_one` and
`assert_probabilities_valid` raise rather than warn. In `models_only` mode the
weights are normalised over the five model terms only, per §7.2.

### Input and Output
**Inputs:** `test_cal`, `prev_te`, `ev_te`, the fitted `fusion`.
**Outputs:** `fw` (a `FusionWeights` with `weights`, `reliabilities`,
`fused_proba`, `risk`); `fusion_weights_notebook.csv`,
`fusion_weights_notebook.json`, `fusion_weights.png`.

### Research Methodology Connection
Implements Stage 7.2 equation (24) and Stage 7.6 equation (27).

### Important Considerations
- **Deviation (D19):** equation (24) adds `w_i·P̂_i` (a class distribution) to
  `w_S·SP_t` and `w_T·TC_t` (scalars); the scalar reading of a model's evidence
  is documented above.
- The per-session weight matrix is saved so the adaptivity claim can be checked
  directly: a model whose weight never varies across sessions is not contributing
  adaptively, whatever the mean weight suggests.


In [ ]:
fw = fusion.fuse(test_cal, prev_state=prev_te,
                 sp=ev_te["sp"] if ev_te else None,
                 tc=ev_te["tc"] if ev_te else None,
                 availability=ev_te["availability"] if ev_te else None)

utils.assert_weights_sum_to_one(fw.weights, "fusion weights")
utils.assert_probabilities_valid(fw.fused_proba, "fused probabilities")
print("[OK] weights sum to 1 per session; fused probabilities sum to 1 per row\n")

fus.save_fusion_artifacts(fw, fusion, OUT, "notebook")
pd.DataFrame({"source": fw.sources,
              "mean weight": fw.weights.mean(axis=0).round(4),
              "std weight": fw.weights.std(axis=0).round(4),
              "min": fw.weights.min(axis=0).round(4),
              "max": fw.weights.max(axis=0).round(4)})

In [ ]:
ev.plot_fusion_weights(fw, cfg, OUT / "fusion_weights.png")
display(Image(str(OUT / "fusion_weights.png")))

## 15. Single-Session Worked Example — Stage 7.4

### Purpose
Make the weighting mechanism legible on one concrete session, in the layout of
the methodology's own illustrative table.

### What Happens in This Cell
The highest-risk test row is selected, and its baseline `Q`, confidence `C`,
reliability `q`, weight `w` and attack score are tabulated per source, together
with `R_t`, `τ_R`, the alert decision and the fused class.

### Method / Algorithm
Per-source decomposition of equations (25), (26) and (24) for a single index.

### Code Logic
`np.argmax(fw.risk)` picks the row. `fus.confidence_term` recomputes `C_i(t)`
for display. The `w × score` column sums to that row's `R_t`.

### Input and Output
**Inputs:** `fw`, `fusion`, `test_cal`, `ev_te`, `y_te`. **Output:** a rendered
table.

### Research Methodology Connection
Reproduces the §7.4 table structure from fitted values.

### Important Considerations
- **Deviation (D21):** §7.4 states its own numbers are illustrative. The values
  here are real, so they will not match the document's, and they should not be
  quoted as agreeing with it.


In [ ]:
# A single-session worked example, in the layout of methodology section 7.4.
# (The document's own numbers there are illustrative; these are real.)
row = int(np.argmax(fw.risk))
print(f"worked example - test row {row}, "
      f"true class = {CLASS_NAMES[int(y_te[row])]}, "
      f"R_t = {fw.risk[row]:.4f}, tau_R = {fusion.tau_R:.4f} "
      f"-> {'ALERT' if fw.risk[row] >= fusion.tau_R else 'no alert'}")

detail = []
for j, src in enumerate(fw.sources):
    entry = {"source": src,
             "baseline Q": round(fusion.Q.get(src, float('nan')), 4),
             "reliability q": round(float(fw.reliabilities[row, j]), 4),
             "weight w": round(float(fw.weights[row, j]), 4)}
    if src in test_cal:
        entry["confidence C"] = round(float(fus.confidence_term(test_cal[src])[row]), 4)
        entry["attack score 1-P(Benign)"] = round(float(1 - test_cal[src][row, 0]), 4)
    elif src == "sp":
        entry["attack score 1-P(Benign)"] = round(float(ev_te["sp"][row]), 4)
    elif src == "tc":
        entry["attack score 1-P(Benign)"] = round(float(ev_te["tc"][row]), 4)
    detail.append(entry)
display(pd.DataFrame(detail))
print(f"\nfused class = {CLASS_NAMES[int(np.argmax(fw.fused_proba[row]))]}")

## 16. Fusion Ablation Baselines — Stage 7.6

### Purpose
Test whether the adaptive mechanism earns its complexity, by scoring simpler
weighting schemes on the same predictions.

### What Happens in This Cell
`fus.run_fusion_ablations` evaluates the §7.6 variants and writes
`fusion_ablations.csv`.

### Method / Algorithm
The §7.6 baselines: `single_best` (the highest-`Q` classifier alone);
`equal_weights` (`w_i = 1/m`); `static_weights` (λ = (1, 0, 0), so weights depend
only on validation macro-F1 and do not change per session); `no_confidence`
(λ₂ = 0) and `no_context` (λ₃ = 0) in turn; `no_temporal` (`SP` and `TC` removed,
leaving the five-model fusion); and `proposed`.

### Code Logic
Each variant re-runs `fusion.fuse` with modified λ or sources on the **already
computed** calibrated probabilities, so no model is retrained and the comparison
isolates the weighting scheme.

### Input and Output
**Inputs:** `fusion`, `test_cal`, `y_te`, `prev_te`, `ev_te`.
**Config keys:** `fusion.ablations.enabled`, `fusion.ablations.variants`.
**Output:** `fusion_ablations.csv`.

### Research Methodology Connection
Implements §7.6, which requires these be reported as ablations only, never as
alternative methods.

### Important Considerations
- **Evaluation validity:** if `proposed` does not beat `static_weights` and
  `equal_weights`, the adaptive mechanism is not earning its place on this data,
  and the ablation table is the evidence either way. It is printed alongside the
  headline result for exactly that reason.
- This cell is disabled for the Cell 22 leakage-gap runs, because it is computed
  after the fused probabilities exist and cannot change them.


In [ ]:
ablations = fus.run_fusion_ablations(
    fusion, test_cal, y_te, prev_te,
    ev_te["sp"] if ev_te else None, ev_te["tc"] if ev_te else None,
    ev_te["availability"] if ev_te else None, cfg)
utils.save_table(ablations, OUT / "fusion_ablations.csv")
print("Stage 7.6 ablations (baselines, not alternative methods):")
ablations

## 8. Evaluation — Stage 8

Macro-F1 and per-class recall are the primary metrics; accuracy is dominated by
the benign class.

## 17. Test-Set Evaluation — Stage 8.1

### Purpose
Score every model and the fusion on the held-out test partition with the
imbalance-aware metrics the methodology designates as primary.

### What Happens in This Cell
The per-model, per-calibrated-model and fused probability matrices are collected
into `proba_for_eval`, and `ev.full_evaluation` computes metrics, per-class
results, significance tests and plots. The per-class table for the fusion is then
displayed with the rare classes flagged.

### Method / Algorithm
§8.1 metric groups: figure metrics (accuracy, precision, recall, F1, AUC);
imbalance-aware metrics (macro-F1, weighted F1, per-class recall, macro
one-vs-rest ROC-AUC, PR-AUC); calibration (ECE, Brier); efficiency.

### Code Logic
`evaluate_predictions` wraps each sklearn metric so a degenerate case returns NaN
instead of aborting the run, and flags classes listed in
`evaluation.rare_classes`. The metric table is sorted by macro-F1 descending, so
`metrics.iloc[0]` is the best label — a convention Cell 22 relies on.

### Input and Output
**Inputs:** `y_te`, `test_proba`, `test_cal`, `fw.fused_proba`.
**Outputs:** `report`; `metrics_notebook.csv`, `metrics_per_class_notebook.csv`,
`metrics_notebook.json`, `significance_notebook.csv`, confusion matrices, ROC and
PR curves.

### Research Methodology Connection
Implements Stage 8.1. Accuracy alone is explicitly insufficient because the
benign class dominates it; macro-F1 and per-class recall are the primary metrics.

### Important Considerations
- **Evaluation validity:** per-class recall for Heartbleed (11 records) and
  Infiltration (36) is estimated from a handful of test flows and will be noisy
  across seeds. Multi-seed runs (`seed.enable_multi_seed`) are the honest way to
  report them.
- The test partition is touched here for the first time; every fitted quantity
  was frozen before this cell.


In [ ]:
proba_for_eval = dict(test_proba)
proba_for_eval["fusion"] = fw.fused_proba
for n, p in test_cal.items():
    proba_for_eval[f"{n}_calibrated"] = p

report = ev.full_evaluation(y_te, proba_for_eval, CLASSES, cfg, OUT,
                            CLASS_NAMES, tag="notebook", make_plots=True,
                            extra_stats=stats)
report["metrics"][["label", "accuracy", "macro_f1", "weighted_f1",
                   "recall_macro", "roc_auc_ovr_macro", "pr_auc_macro",
                   "ece", "brier", "rare_class_recall_macro"]].round(4)

In [ ]:
print("per-class results for the fusion (rare classes flagged):")
pc = report["per_class"]
pc[pc["label"] == "fusion"][["class", "support", "precision", "recall", "f1", "is_rare"]].round(4)

## 18. Diagnostic Figures and Paired Significance Testing — Stage 8.1 and 8.3

### Purpose
Render the confusion, ROC and PR diagnostics, and establish whether the
differences between models are statistically distinguishable rather than noise.

### What Happens in This Cell
Saved figures are displayed inline, then the pairwise significance table is shown
when non-empty.

### Method / Algorithm
McNemar's test on the paired correct/incorrect disagreement counts, using the
exact binomial when the discordant count is small and the chi-square
approximation with continuity correction otherwise; a paired bootstrap over the
macro-F1 difference is available as an alternative.

### Code Logic
`compare_models` builds every pair from the prediction vectors;
`evaluation.significance.test` selects the test and `alpha` the threshold.

### Input and Output
**Inputs:** `report`, saved PNGs. **Output:** `significance_notebook.csv`.

### Research Methodology Connection
Implements the §8.3 requirement for significance tests where models are compared.

### Important Considerations
- **Evaluation validity:** McNemar tests agreement on individual predictions, not
  the macro-F1 difference. A pair can differ significantly on per-sample
  correctness while their macro-F1 values are close, and vice versa; the
  bootstrap variant tests the macro-F1 difference directly.


In [ ]:
best = report["metrics"].iloc[0]["label"]
for p in [OUT / f"confusion_fusion_notebook.png",
          OUT / f"confusion_fusion_norm_notebook.png",
          OUT / "roc_curves_notebook.png",
          OUT / "pr_curves_notebook.png"]:
    if p.exists():
        display(Image(str(p)))

In [ ]:
if len(report["significance"]):
    print("pairwise significance tests "
          f"({cfg['evaluation']['significance']['test']}):")
    display(report["significance"].head(15).round(5))

## 19. SHAP Attribution for the Selected Features — Stage 4.4 (FeS Set-4)

### Purpose
Show which features the SHAP-LGBM ranker attributed the model output to, as
interpretability evidence for the selection.

### What Happens in This Cell
`ev.plot_shap_summary` renders the mean |SHAP| bar chart from the already-computed
`fs_result.ranker_scores["shap_lgbm"]`, and the top features are tabulated.

### Method / Algorithm
Equation (6): `SFI(f_j) = (1/N) Σ_i |φ_ij|`, aggregated over rows **and** classes
for the multi-class model.

### Code Logic
No SHAP values are recomputed here; the scores come from the Stage 4 cache, so
this cell is inexpensive.

### Input and Output
**Inputs:** `fs_result.ranker_scores`, `fs_result.f_corr`.
**Output:** `shap_summary.png`.

### Research Methodology Connection
Visualises FeS Set-4 of §4.4 and its §4.5 specification.

### Important Considerations
- This is one ranker of eight. A feature ranked highly here but absent from `F*`
  simply failed to reach the vote threshold, which is the intended behaviour of
  the fusion and not an inconsistency. See Cell 6.


In [ ]:
# SHAP-LGBM ranker summary (FeS Set-4)
if "shap_lgbm" in fs_result.ranker_scores:
    ev.plot_shap_summary(fs_result.ranker_scores["shap_lgbm"], fs_result.f_corr,
                         cfg, OUT / "shap_summary.png")
    display(Image(str(OUT / "shap_summary.png")))
    top = np.argsort(-fs_result.ranker_scores["shap_lgbm"])[:12]
    display(pd.DataFrame({"feature": [fs_result.f_corr[i] for i in top],
                          "mean |SHAP|": fs_result.ranker_scores["shap_lgbm"][top]}).round(5))

## 9. Temporal pattern mining — Stage 9

Mined on TRAIN sequences only. See README D15–D19 for the choices this stage
makes, since the methodology names it but does not specify it.

## 20. Mined Patterns and the Attack-State Graph — Stage 9

### Purpose
Present the Stage 9 artifacts — the frequent itemsets, the ordered subsequences,
and the transition structure that supplies `K_i(t)` and `TC_t`.

### What Happens in This Cell
Counts and the top patterns are printed, the pattern bar chart and the graph are
drawn, the transition matrix is displayed, and the `SP_t`, `TC_t` and `R_t`
distributions are histogrammed with `τ_R` marked.

### Method / Algorithm
- **FP-Growth** mines frequent *unordered* itemsets over the session event sets.
- **PrefixSpan** mines frequent *ordered* subsequences by recursive
  projected-database pattern growth, which is what distinguishes it from
  FP-Growth.
- **Attack-state graph `G_T`:** nodes are behavioural states, edge weights are
  Laplace-smoothed transition probabilities estimated on TRAIN.

### Code Logic
`_is_subsequence` allows gaps but enforces order, so `a → c` matches
`a, b, c` while `c → a` does not. `AttackStateGraph.consistency` scales the mean
transition probability by the number of states and clips to [0, 1].

### Input and Output
**Inputs:** `temporal`, `ev_te`, `fw.risk`, `fusion.tau_R`.
**Outputs:** `top_patterns.png`, `attack_state_graph.png`,
`temporal_distributions.png`; the CSV and GraphML written in Cell 13.

### Research Methodology Connection
Implements the Stage 9 extension, which §0 notes is part of the research but is
not drawn in the methodology figure.

### Important Considerations
- The fraction of sessions with temporal evidence available is printed: sessions
  below `fusion.min_events_for_temporal` events contribute no `SP`/`TC` and
  receive zero weight on those sources, so a low availability fraction means the
  temporal layer is contributing to few decisions.
- Pattern support depends on the session reconstruction of Cell 1; a different
  `session_key_mode` or idle timeout changes the event sequences and therefore
  every Stage 9 quantity.


In [ ]:
print(f"FP-Growth itemsets : {len(temporal.fp_itemsets)}")
print(f"PrefixSpan patterns: {len(temporal.sequences_patterns)}")
print(f"attack-state graph : {len(temporal.graph.states)} states, "
      f"{int(temporal.graph.counts.sum())} transitions observed")
display(temporal.sequences_patterns.head(12)[["pattern", "length", "count", "support"]])

In [ ]:
for fn, path in [
    (lambda: ev.plot_top_patterns(temporal.sequences_patterns, cfg, OUT / "top_patterns.png"),
     OUT / "top_patterns.png"),
    (lambda: ev.plot_attack_state_graph(temporal.graph, cfg, OUT / "attack_state_graph.png"),
     OUT / "attack_state_graph.png"),
]:
    fn()
    if path.exists():
        display(Image(str(path)))

In [ ]:
# Transition matrix of G_T
tm = pd.DataFrame(temporal.graph.matrix, index=temporal.graph.states,
                  columns=temporal.graph.states)
display(tm.round(3).iloc[:10, :10])

if ev_te is not None:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    axes[0].hist(ev_te["sp"], bins=30, color="#7a5195"); axes[0].set_title("$SP_t$ (test)")
    axes[1].hist(ev_te["tc"], bins=30, color="#2a7f3f"); axes[1].set_title("$TC_t$ (test)")
    axes[2].hist(fw.risk, bins=30, color="#d1495b")
    axes[2].axvline(fusion.tau_R, color="k", ls="--", label=f"$\\tau_R$={fusion.tau_R:.2f}")
    axes[2].set_title("fused risk $R_t$ (test)"); axes[2].legend()
    for a in axes:
        a.grid(alpha=0.3)
    fig.tight_layout()
    utils.save_fig(fig, OUT / "temporal_distributions.png", cfg["run"]["figure_dpi"], close=False)
    plt.show()
    print(f"sessions with temporal evidence available: "
          f"{ev_te['availability'].mean():.1%}")

## 10. Branch comparison and the leakage gap

Branch D's point: a plain flow-level stratified split tears sessions apart and
puts near-duplicate flows on both sides. The drop from that number to the
time-blocked number is how much of a reported score comes from the evaluation
protocol rather than from the detector.

### A note on what these three runs compute

The comparison below re-runs the **whole pipeline** under three evaluation
protocols on identical data, and compares one number: the fusion macro-F1 on the
test partition. Two pieces of work are therefore switched off for these runs
only, via `GAP_OVERRIDES`:

* `feature_selection.stability.enabled` — Stage 8.3 selection stability is a
  diagnostic over repeated stratified subsamples. It is computed *after* `F*` is
  fixed and never alters it, so it cannot change any metric compared here.
* `fusion.ablations.enabled` — the Stage 7.6 baselines (`single_best`,
  `equal_weights`, `static_weights`, …) are scored *after* the fused
  probabilities exist and never alter them.

Everything that can affect the compared metric is left exactly as configured and
is **identical across the three protocols**: the eight rankers, the vote
threshold η, the `N*` rule, the imbalance mode, all five models, calibration,
the fusion weights and `τ_R`. Only the split strategy varies — which is the
point of the comparison.

Both switches stay on for the main run in cells 1–9 and for `run_all.py`, so the
thesis results still carry the stability table and the ablation table.

The time-blocked protocol is **not** re-run here: cells 1–9 already executed it,
and their `report`, `fs_result` and `split` are reused. Each strategy is also
resumable — if `results/notebook/<strategy>/` already holds its saved metrics
and `run.use_cache` is true, they are loaded instead of recomputed.


## 21. Leakage Gap Across Three Evaluation Protocols — Stage 3.2 and 8.4

### Purpose
Quantify how much of a reported score comes from the evaluation protocol rather
than from the detector, by running the identical pipeline three ways on identical
data.

### What Happens in This Cell
For each of `stratified_flow`, `stratified_session` and `time_blocked` the cell
either reuses the main run from Cells 1–9, loads a previous run's saved metrics,
or runs the full pipeline; it then computes the gap against the time-blocked
result and writes `leakage_gap.csv`.

### Method / Algorithm
- `stratified_flow` — plain flow-level stratified split. Sessions are torn apart,
  so near-duplicate flows of one session land on both sides. This is the protocol
  most published CIC-IDS2017 numbers use.
- `stratified_session` — stratified but session-grouped.
- `time_blocked` — the methodology's main protocol.
The **leakage gap** is the drop from the flow-level number to the time-blocked
number.

### Code Logic
`GAP_OVERRIDES` disables `feature_selection.stability` and `fusion.ablations` for
these runs only; both are computed after the quantities they would affect are
fixed, and both are applied identically to all three protocols, so the comparison
remains fair. `_reuse_main_run` returns the Cells 1–9 result when the strategy
matches `cfg["split"]["strategy"]`; `_load_cached` resumes from
`results/notebook/<strategy>/metrics_<strategy>.csv` and `summary.json`. The
`source` column records which path supplied each row.

### Input and Output
**Inputs:** `df_cache`, `cfg`, and the Cells 1–9 objects `report`, `fs_result`,
`split`.
**Outputs:** `gap`; `results/notebook/leakage_gap.csv`, `leakage_gap.png`, and
per-strategy subdirectories.

### Research Methodology Connection
Implements the §3.2 secondary protocols and the §8.4 statement that whole
sessions stay on one side of the split — here measured rather than asserted.

### Important Considerations
- **Evaluation validity:** a near-zero gap is only meaningful on data with real
  intra-session redundancy. On synthetic data whose flows are drawn
  independently, no gap can appear and none does; this does not refute the
  premise.
- **Runtime cost:** reuse and caching mean a full re-execution costs at most two
  pipeline runs instead of three, and neither runs the stability diagnostic.


In [ ]:
import time

# Work the leakage-gap comparison does not use. Applied IDENTICALLY to all three
# protocols, so the comparison stays fair; see the note above for why neither can
# affect the fusion macro-F1 that is compared.
GAP_OVERRIDES = {
    "feature_selection": {"stability": {"enabled": False}},
    "fusion": {"ablations": {"enabled": False}},
}
GAP_STRATEGIES = ["stratified_flow", "stratified_session", "time_blocked"]

df_cache = (df, feature_columns, prep_report)


def _gap_row(strategy, metrics, n_star, sizes, source):
    """Build one comparison row, whatever supplied the metrics frame."""
    fus = metrics.loc[metrics["label"] == "fusion"]
    return {
        "split_strategy": strategy,
        "fusion_macro_f1": float(fus["macro_f1"].iloc[0]) if len(fus) else None,
        "fusion_accuracy": float(fus["accuracy"].iloc[0]) if len(fus) else None,
        "best_macro_f1": float(metrics.iloc[0]["macro_f1"]),
        "n_star": n_star,
        **{f"n_{k}": v for k, v in sizes.items()},
        "source": source,
    }


def _reuse_main_run(strategy):
    """Cells 1-9 already ran the time-blocked protocol -- reuse that result."""
    if strategy != cfg["split"]["strategy"]:
        return None
    if not all(v in globals() for v in ("report", "fs_result", "split")):
        return None
    return _gap_row(strategy, report["metrics"], fs_result.n_star,
                    split.sizes(), "main run (cells 1-9)")


def _load_cached(strategy):
    """Resume: load a previous run's saved metrics for this strategy."""
    d = OUT / strategy
    mfile, sfile = d / f"metrics_{strategy}.csv", d / "summary.json"
    if not (mfile.exists() and sfile.exists()):
        return None
    try:
        summary = json.loads(sfile.read_text())
        return _gap_row(strategy, pd.read_csv(mfile), summary["n_star"],
                        summary["split_sizes"], "cache")
    except Exception as exc:
        print(f"[{strategy}] cache unreadable ({exc}); recomputing")
        return None


gap_rows, est_min = [], None
for strategy in GAP_STRATEGIES:
    row = _reuse_main_run(strategy)
    if row is not None:
        print(f"[{strategy}] reusing the main time-blocked run from cells 1-9 "
              f"(no recompute)")
    else:
        row = _load_cached(strategy) if cfg["run"].get("use_cache", True) else None
        if row is not None:
            print(f"[{strategy}] loaded from cache -> {OUT / strategy}")
        else:
            est = (f"~{est_min:.0f} min (from the previous strategy)"
                   if est_min else "unknown on the first strategy")
            print(f"[{strategy}] running the full pipeline - estimated {est}")
            t0 = time.perf_counter()
            c = utils.deep_merge(
                utils.deep_merge(cfg, GAP_OVERRIDES),
                {"split": {"strategy": strategy},
                 "run": {"name": f"nb_{strategy}",
                         "output_dir": str(OUT / strategy)}})
            r = pipeline.run_pipeline(DATA_DIR, cfg=c, out_dir=c["run"]["output_dir"],
                                      tag=strategy, make_plots=False, df_cache=df_cache)
            est_min = (time.perf_counter() - t0) / 60
            print(f"[{strategy}] finished in {est_min:.1f} min")
            row = _gap_row(strategy, r.metrics, r.feature_result.n_star,
                           r.split.sizes(), "computed")
    gap_rows.append(row)

gap = pd.DataFrame(gap_rows)
ref = gap.loc[gap["split_strategy"] == "time_blocked", "fusion_macro_f1"].iloc[0]
gap["leakage_gap_vs_time_blocked"] = gap["fusion_macro_f1"] - ref
utils.save_table(gap, OUT / "leakage_gap.csv")
gap.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
colors = ["#d1495b", "#edae49", "#2a7f3f"]
ax.bar(gap["split_strategy"], gap["fusion_macro_f1"], color=colors)
for i, v in enumerate(gap["fusion_macro_f1"]):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)
ax.set_ylabel("fusion macro-F1 (test)")
ax.set_title("Leakage gap: the same pipeline under three evaluation protocols")
ax.grid(axis="y", alpha=0.3)
utils.save_fig(fig, OUT / "leakage_gap.png", cfg["run"]["figure_dpi"], close=False)
plt.show()

print(f"The flow-level protocol reports "
      f"{gap.iloc[0]['leakage_gap_vs_time_blocked']:+.4f} macro-F1 relative to "
      f"the time-blocked protocol on identical data.")

## 22. Aggregated Branch Comparison — Stage 8.2 (branch level)

### Purpose
Surface the cross-branch comparison produced by the command-line driver, which
covers the imbalance-strategy branches and the zero-day branch that the notebook
itself does not run.

### What Happens in This Cell
If `results/branch_comparison.csv` exists it is displayed; otherwise the command
that produces it is printed.

### Code Logic
`run_all.py` writes that file after running the root pipeline and branches A–E.
Branch A is class weighting only, B is SMOTE-ENN only, C is both (the notebook's
configuration), D is the stratified-split comparison, and E is leave-one-attack-out.

### Input and Output
**Input:** `results/branch_comparison.csv`. **Output:** a rendered table or an
instruction.

### Research Methodology Connection
Connects the notebook to the branch experiments of Stage 5.2 (the three imbalance
branches) and the §3.2 optional leave-one-attack-out protocol.

### Important Considerations
- **Bug (fixed):** the printed command previously named `--smoke_test`, while
  `run_all.py` now takes `--smoke`.


In [ ]:
# If run_all.py has been executed, show the aggregated branch comparison.
comparison = Path("results/branch_comparison.csv")
if comparison.exists():
    display(pd.read_csv(comparison))
else:
    print("Run `python run_all.py --data_dir <path> --smoke` to produce "
          "results/branch_comparison.csv across all five branches.")

## Stage checklist

## 23. Stage Completion Checklist — Stage all

### Purpose
Verify that every methodology stage actually produced its artifact in this run,
rather than inferring completion from the absence of an exception.

### What Happens in This Cell
Each stage is paired with its implementing module and an expected output path;
existence is checked by glob and a stage → file → status table is printed, with
anything missing named explicitly.

### Code Logic
Artifacts carry the run tag in their filename (for example
`metrics_notebook.csv`), so each check is a glob rather than an exact name.
`utils.format_checklist` renders the fixed-width table.

### Input and Output
**Input:** the contents of `results/notebook/`. **Output:** printed table.

### Research Methodology Connection
Provides the end-to-end evidence that Stages 1–9 ran, which is the final
requirement of the implementation plan.

### Important Considerations
- A `MISSING` row means that stage produced nothing in *this* run — for example
  because a configuration flag disabled it — and is not by itself a failure, but
  it must be explained before the run is cited as complete.


In [ ]:
checks = [
    ("Stage 1: dataset consolidation", "data_preprocessing.py", OUT / "class_distribution.csv"),
    ("Stage 2: preprocessing",         "data_preprocessing.py", OUT / "preprocessing_report.json"),
    ("Stage 3: splits",                "data_preprocessing.py", OUT / "split_distribution.csv"),
    ("Stage 4: feature selection",     "feature_selection.py",  OUT / "feature_votes*.csv"),
    ("Stage 5: imbalance handling",    "imbalance_handling.py", OUT / "imbalance_report.csv"),
    ("Stage 6: models",                "models.py",             OUT / "model_efficiency.csv"),
    ("Stage 7: evidence fusion",       "fusion.py",             OUT / "fusion_weights*.csv"),
    ("Stage 8: evaluation",            "evaluation.py",         OUT / "metrics*.csv"),
    ("Stage 9: temporal mining",       "temporal_pattern_mining.py", OUT / "attack_state_graph.json"),
    ("Leakage gap (branch D)",         "branches/branch_D_stratified_split/run.py", OUT / "leakage_gap.csv"),
]
# Artifacts carry the run tag in their name, so each entry is a glob.
rows = [(s, f, "OK" if sorted(Path(a).parent.glob(Path(a).name)) else "MISSING")
        for s, f, a in checks]
print(utils.format_checklist(rows))
missing = [r[0] for r in rows if r[2] == "MISSING"]
print(f"\nNot produced by this run: {missing}" if missing else "\nEvery stage produced its artifacts.")